# E5 — Mixture-composition recognition

Predict the presence of acetone, isopropanol, and pentane (eight compositions including air) without concentrations at inference. The primary test is the later campaign 146, with diagnostic analyses to distinguish missed mixtures from wrong component combinations.

In [ ]:
from pathlib import Path
import os, sys, re, json, random, copy, time, hashlib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

ROOT = Path.cwd().resolve()
SRC = next((q for p in (ROOT, *ROOT.parents) for q in (p, p/'src')
            if (q/'esdata').is_dir()), None)
if SRC is None:
    raise FileNotFoundError('Run from the ml_flavored-ml repository (root or src/)')
sys.path.insert(0, str(SRC))
from esdata import EsmellDataset, Chain
from esdata.transforms import (DefaultCSVPreprocessing, SimplifyLabels, SelectRChannels,
    UserLabelCycleSegments, CoolingHeatingSegments, CanonicalTempAlign,
    CanonicalSubtractionFeatures)

DATA_DIR = Path(os.getenv('ESMELL_DATA_DIR', str(SRC.parent / 'data' / 'raw')))
OUT = SRC / 'outputs' / 'E5_diagnostic_v2'
OUT.mkdir(parents=True, exist_ok=True)
ARCHIVES = {130: 'curiosity-2v15.130.0.tar.gz',
            146: 'curiosity-2v15.146.0.tar.gz'}
GASES = ('acetone', 'isopropanol', 'pentane')
CHANNELS = ('R1', 'R2', 'R11', 'R12', 'R16', 'R17')
SEEDS = (42, 43, 44)
METHODS = ('ExtraTrees3', 'ExtraTrees8', 'ROCKET', 'CNN', 'ConvTran')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_EPOCHS, PATIENCE, BATCH = 70, 12, 32
RUN_SAME_DAY = True    # independent train-only holdout (fixed threshold)
RUN_MEMORIZATION = True  # 32-example optimization sanity probe for CNN/ConvTran
RUN_WEIGHT_ABLATION = True  # soft versus balanced weights, two models
USE_CACHE = True
CACHE_FILE = OUT / 'signals_cache.npz'

# The two labels are subsets of the same three gases. Bit 0=acetone, bit 1=isopropanol, bit 2=pentane.
BITS = ((np.arange(8)[:, None] >> np.arange(3)[None, :]) & 1).astype(np.uint8)

def locate_archive(filename):
    direct = DATA_DIR / filename
    if direct.is_file(): return direct.name
    stem = filename.removesuffix('.tar.gz')
    candidates = sorted(DATA_DIR.glob(stem + '*.tar*.gz'))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one archive matching {stem} in {DATA_DIR}, found {candidates}')
    return candidates[0].name

if not DATA_DIR.is_dir():
    raise FileNotFoundError('Set ESMELL_DATA_DIR to the directory containing the 130 and 146 archives')
ARCHIVES = {card: locate_archive(name) for card, name in ARCHIVES.items()}
print('Device:', DEVICE, '\nArchives:', ARCHIVES, '\nOutput:', OUT)
# Fail early rather than after extracting thousands of signals.
try:
    from sktime.transformations.panel.rocket import Rocket
except ImportError as e:
    raise ImportError('ROCKET requires sktime and numba; install with pip install sktime numba') from e

## 1. Records and integrity

Use fixed train/validation/test periods from campaigns 130 and 146 and the canonical 12 × 200 heating/hysteresis response.

In [ ]:
LABEL = re.compile(r'^(acetone|isopropanol|pentane)\[([0-9]+(?:\.[0-9]+)?)ppb\]$')

def components(text):
    s = str(text).strip().lower()
    if s in ('air', 'none'):
        return np.zeros(3, dtype=np.uint8)
    parts = s.split('+')
    parsed = [LABEL.fullmatch(p) for p in parts]
    if not parsed or any(x is None for x in parsed):
        raise ValueError(f'Unknown/malformed gas label: {text!r}')
    names = [x.group(1) for x in parsed]
    if len(set(names)) != len(names):
        raise ValueError(f'Duplicate gas in {text!r}')
    return np.array([int(g in names) for g in GASES], dtype=np.uint8)


def load_card(card):
    source = EsmellDataset(root_path=DATA_DIR, subfolders=ARCHIVES[card],
                           csv_repair='strict', archive_cache='disk', preload=True)
    pipe = source >> Chain(DefaultCSVPreprocessing(), SimplifyLabels(),
        SelectRChannels(include=CHANNELS), UserLabelCycleSegments(),
        CoolingHeatingSegments(temp_col='T_mean', smooth_window=3, eps=.2),
        CanonicalTempAlign(n_grid=200, temp_range=(175., 390.),
                           normalize=False, conductance=True),
        CanonicalSubtractionFeatures(features=((1, -3, 5), ((1, -3, 5), (0, -2, 4)))))
    rows, tensors = [], []
    for sid in source.ids:
        info = source.metadata(sid)
        stamp = pd.Timestamp(info['datetime'])
        if card == 130:
            split = 'train' if stamp < pd.Timestamp('2026-09-28') else (
                    'validation' if stamp < pd.Timestamp('2026-09-29') else 'excluded')
        else:
            split = 'test'
        label = components(info['classname'])
        row = dict(sample_id=str(sid), filename=info['filename'], card=card,
                   recipe_key='+'.join(sorted(str(info['classname']).lower().split('+'))),
                   time=stamp, split=split, recipe=str(info['classname']),
                   composition=int(label @ (1 << np.arange(3))))
        rows.append(row)
        if split == 'excluded':
            continue
        signal = np.asarray(pipe.features(sid), dtype=np.float32)
        if signal.shape != (12, 200) or not np.isfinite(signal).all():
            raise ValueError(f'Invalid sensor signal for {sid}: {signal.shape}')
        tensors.append((str(sid), signal))
    return pd.DataFrame(rows), dict(tensors)

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4<<20),b''):h.update(block)
    return h.hexdigest()
archive_hashes={card:file_sha256(DATA_DIR/name) for card,name in ARCHIVES.items()}
# Cache invalidates if archive bytes or transformation code change, not merely filenames.
code_hashes={str(p.relative_to(SRC)):file_sha256(p) for p in (SRC/'esdata').rglob('*.py')}
pipeline_signature = json.dumps(dict(archive_hashes=archive_hashes,code_hashes=code_hashes,channels=CHANNELS,
    profile='DefaultCSV+Simplify+SelectR+Cycles+CoolHeat3eps.2+CanonicalTempAlign200(175,390)+Subtraction1,-3,5'),sort_keys=True)
cache_signature = hashlib.sha256(pipeline_signature.encode()).hexdigest()
if USE_CACHE and CACHE_FILE.is_file():
    with np.load(CACHE_FILE, allow_pickle=False) as cc:
        if str(cc['signature'].item()) != cache_signature:
            raise RuntimeError('Cached signals belong to another archive/channel/pipeline; delete signals_cache.npz')
        cached_meta = pd.read_json(str(cc['metadata_json'].item()),orient='records')
        cached_meta['time']=pd.to_datetime(cached_meta['time'])
        X_cached=cc['X'].astype(np.float32)
    meta130=cached_meta[cached_meta.card.eq(130)].copy()
    meta146=cached_meta[cached_meta.card.eq(146)].copy()
    print('Loaded verified feature cache:',CACHE_FILE)
else:
    meta130, signals130 = load_card(130)
    meta146, signals146 = load_card(146)
meta = pd.concat([meta130, meta146], ignore_index=True)
meta.to_csv(OUT/'split_manifest.csv', index=False)

usable = meta[meta.split != 'excluded'].sort_values(['card','time']).reset_index(drop=True)
if 'X_cached' in globals():
    # cache records have the exact same ordering as the stored rows
    assert len(X_cached)==len(usable)
    X=X_cached
else:
    X=np.stack([signals130[r.sample_id] if r.card==130 else signals146[r.sample_id]
                for r in usable.itertuples()])
    if USE_CACHE:
        np.savez_compressed(CACHE_FILE,signature=np.array(cache_signature),
            metadata_json=np.array(meta.to_json(orient='records',date_format='iso')),
            X=X)
recipe = usable.composition.to_numpy(dtype=np.int64)
Y = BITS[recipe].astype(np.float32)
tr = np.flatnonzero(usable.split.eq('train'))
va = np.flatnonzero(usable.split.eq('validation'))
te = np.flatnonzero(usable.split.eq('test'))
assert (len(meta130), len(meta146)) == (2570, 916)
assert (len(tr), len(va), len(te)) == (2409, 152, 916)
assert X.shape == (3477,12,200) and len(set(usable.sample_id)) == len(usable)
assert all(set(recipe[idx]) == set(range(8)) for idx in (tr,va,te))
assert usable.iloc[tr].time.max() < usable.iloc[va].time.min()
assert usable.iloc[va].time.max() < usable.iloc[te].time.min()
assert all((BITS[recipe[idx]].sum(1)>1).sum() == n
           for idx,n in ((tr,154),(va,125),(te,98)))
assert not (set(usable.iloc[tr].sample_id) & set(usable.iloc[te].sample_id))
test_mix = usable.iloc[te].loc[Y[te].sum(1)>1]
assert set(test_mix.recipe_key).issubset(set(usable.iloc[tr].recipe_key)), \
    'Unseen mixture recipes: do not claim seen-recipe generalization'
assert len(set(pd.to_datetime(test_mix.time).dt.date)) == 1

print(pd.crosstab(usable.split, recipe).rename(columns=lambda i: ''.join(map(str,BITS[i]))))
print('Data:', X.shape, '| Train/Val/Test:', len(tr),len(va),len(te))
# Leakage/integrity: independent physical files and no complete file reused across splits
assert usable.filename.is_unique, 'Duplicate filenames: inspect the data ingestion'
assert not any(set(usable.iloc[a].filename)&set(usable.iloc[b].filename)
               for a,b in [(tr,va),(tr,te),(va,te)])
# Feature hash duplicates are an ALARM, not proof of leakage (e.g. zero air signals)
usable['signal_sha256']=[hashlib.sha256(z.tobytes()).hexdigest() for z in X]
collisions=usable.groupby('signal_sha256').filter(lambda z:z.split.nunique()>1)
collisions[['sample_id','split','recipe','signal_sha256']].to_csv(OUT/'cross_split_signal_hashes.csv',index=False)
print('Identical feature arrays across splits:',len(collisions),'(check air/zero profiles first)')

In [ ]:
# All grouping below is metadata-only; no test-dependent filtering or training decisions.
audit=usable.copy()
audit['n_components']=Y.sum(1).astype(int)
audit['day']=pd.to_datetime(audit.time).dt.date.astype(str)
audit['recipe_key']=audit.recipe_key.astype(str)
train_recipe_counts=usable.iloc[tr].recipe_key.value_counts()
audit['train_recipe_support']=audit.recipe_key.map(train_recipe_counts).fillna(0).astype(int)
assert not audit.loc[te].query('n_components >= 2').train_recipe_support.eq(0).any()
audit.groupby(['split','day','n_components']).size().rename('n').to_csv(OUT/'cohort_by_day.csv')
audit.groupby(['split','composition']).size().rename('n').to_csv(OUT/'cohort_by_composition.csv')
audit[['sample_id','card','split','time','recipe','recipe_key','composition',
       'n_components','train_recipe_support','signal_sha256']].to_csv(OUT/'split_audit.csv',index=False)
print(pd.crosstab([audit.split,audit.day],audit.n_components))
print('Exact recipes per split:',audit.groupby('split').recipe_key.nunique().to_dict())
print('Mixtures per split:',audit[audit.n_components>=2].groupby('split').size().to_dict())
print('Previously unseen val mixture recipes:',int(((audit.split=='validation')&(audit.n_components>=2)&
       (audit.train_recipe_support==0)).sum()))
print('New test mixture recipes:',int(((audit.split=='test')&(audit.n_components>=2)&
       (audit.train_recipe_support==0)).sum()))

## 2. Features and metrics

Evaluate eight-composition macro recall and mixture exact match, separately for binary and ternary mixtures. Include simple constant predictors.

In [ ]:
def stats192(A):
    features = [A.mean(2), A.std(2), A.min(2), A.max(2),
                np.median(A,axis=2), np.quantile(np.abs(A),.9,axis=2)]
    features.extend(z.mean(2) for z in np.array_split(A,10,axis=2))
    return np.concatenate(features, axis=1).astype(np.float32)

Z = stats192(X)
assert Z.shape == (len(X),192)
def make_weights(train_idx,mode='balanced'):
    counts=np.bincount(recipe[train_idx],minlength=8)
    if (counts==0).any():raise ValueError('Training subset lacks one or more compositions')
    if mode=='balanced':
        r=len(train_idx)/(8.0*counts)       # 8 groups each weight 1/8
    elif mode=='soft':
        r=np.clip(np.sqrt(len(train_idx)/(8.0*counts)),.5,3.0)
    else: raise ValueError(mode)
    a=r[recipe].astype(np.float32)
    return a/a[train_idx].mean()

def normalize_on_train(train_idx):
    mu=X[train_idx].mean((0,2),keepdims=True)
    sd=X[train_idx].std((0,2),keepdims=True)
    return ((X-mu)/np.maximum(sd,1e-6)).astype(np.float32)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class ConvBlock(nn.Module):
    def __init__(self, a,b,k,drop):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(a,b,k,padding=k//2),nn.BatchNorm1d(b),
            nn.ReLU(),nn.MaxPool1d(2),nn.Dropout(drop))
    def forward(self,x): return self.net(x)

class MixtureCNN(nn.Module):
    """RobustEsmellNet from notebooks/multi_label_exp.ipynb; three outputs."""
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(ConvBlock(12,64,7,.10),
            ConvBlock(64,128,5,.15),ConvBlock(128,256,3,.20))
        self.avg, self.maxp = nn.AdaptiveAvgPool1d(1),nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(nn.Linear(512,256),nn.ReLU(),nn.Dropout(.30),nn.Linear(256,3))
    def forward(self,x):
        z=self.backbone(x)
        return self.head(torch.cat([self.avg(z).squeeze(-1),self.maxp(z).squeeze(-1)],1))

class ConvTran(nn.Module):
    """Compact ConvTran-style tAPE/eRPE, identical E4 architecture except 12 inputs/3 outputs."""
    def __init__(self,channels=12,length=200,emb=16,heads=8,ff=256,dropout=.1):
        super().__init__()
        self.emb,self.heads=emb,heads
        self.embed=nn.Sequential(nn.ZeroPad2d((3,4,0,0)),
            nn.Conv2d(1,emb*4,(1,8)),nn.BatchNorm2d(emb*4),nn.GELU(),
            nn.Conv2d(emb*4,emb,(channels,1)),nn.BatchNorm2d(emb),nn.GELU())
        pos=torch.arange(length).float()[:,None]
        div=torch.exp(torch.arange(0,emb,2).float()*(-np.log(10000.)/emb))
        pe=torch.zeros(length,emb)
        pe[:,0::2]=torch.sin(pos*div*(emb/length))
        pe[:,1::2]=torch.cos(pos*div*(emb/length))
        self.register_buffer('tape',pe[None])
        idx=torch.arange(length)
        self.register_buffer('rel_idx',idx[:,None]-idx[None,:]+length-1)
        self.rel=nn.Parameter(torch.zeros(2*length-1,heads))
        self.q,self.k,self.v=(nn.Linear(emb,emb,bias=False) for _ in range(3))
        self.attn_norm=nn.LayerNorm(emb)
        self.norm1,self.norm2=nn.LayerNorm(emb),nn.LayerNorm(emb)
        self.drop=nn.Dropout(dropout)
        self.ff=nn.Sequential(nn.Linear(emb,ff),nn.ReLU(),nn.Dropout(dropout),
            nn.Linear(ff,emb),nn.Dropout(dropout))
        self.head=nn.Linear(emb,3)
    def forward(self,x):
        z=self.embed(x[:,None]).squeeze(2).transpose(1,2)
        p=self.drop(z+self.tape)
        b,l,e=p.shape
        def split(m):return m.reshape(b,l,self.heads,e//self.heads).transpose(1,2)
        q,k,v=split(self.q(p)),split(self.k(p)),split(self.v(p))
        a=((q@k.transpose(-2,-1))*(e**-.5)).softmax(-1)
        a=a+self.rel[self.rel_idx].permute(2,0,1)[None]
        h=(a@v).transpose(1,2).reshape(b,l,e)
        z=self.norm1(z+self.attn_norm(h))
        z=self.norm2(z+self.ff(z))
        return self.head(z.mean(1))

for net in [MixtureCNN(),ConvTran()]:
    assert net(torch.zeros(2,12,200)).shape==(2,3)
    print(type(net).__name__,sum(p.numel() for p in net.parameters()),'parameters')

In [ ]:
from scipy.special import expit
from itertools import product

def bits_to_id(a):return np.asarray(a,dtype=int) @ np.array([1,2,4])

def metrics(yt,yp):
    yt=np.asarray(yt,dtype=np.uint8);yp=np.asarray(yp,dtype=np.uint8)
    assert yt.shape==yp.shape and yt.ndim==2 and yt.shape[1]==3
    n=yt.sum(1); exact=(yt==yp).all(1)
    a=bits_to_id(yt);b=bits_to_id(yp)
    cm=confusion_matrix(a,b,labels=np.arange(8))
    denom=cm.sum(1); rec=np.diag(cm)/np.maximum(denom,1)
    macro8=float(rec[denom>0].mean())
    z=dict(n=len(yt),n_mix=int((n>=2).sum()),macro8=macro8,
           exact_all=float(exact.mean()),
           exact_mix=float(exact[n>=2].mean()) if (n>=2).any() else np.nan,
           exact_double=float(exact[n==2].mean()) if (n==2).any() else np.nan,
           exact_triple=float(exact[n==3].mean()) if (n==3).any() else np.nan,
           pred_mix_rate=float((yp.sum(1)>=2).mean()),
           macro_bacc=float(np.nanmean([balanced_accuracy_score(yt[:,k],yp[:,k])
                   if np.unique(yt[:,k]).size==2 else np.nan for k in range(3)])),
           macro_f1=float(f1_score(yt,yp,average='macro',zero_division=0)))
    return z

def gas_discrimination(yt,score):
    output={}
    for j,g in enumerate(GASES):
        v=yt[:,j].astype(int);s=score[:,j]
        output[g+'_auc']=float(roc_auc_score(v,s)) if np.unique(v).size==2 else np.nan
        output[g+'_ap']=float(average_precision_score(v,s)) if v.sum() else np.nan
    return output

def per_bit_thresholds(yt,sc):
    t=[]
    for j in range(3):
        v=yt[:,j].astype(int);s=sc[:,j]
        cand=np.unique(np.r_[.5,np.quantile(s,np.linspace(.05,.95,19))])
        if np.unique(v).size!=2: t.append(.5);continue
        ranks=np.array([balanced_accuracy_score(v,s>=x) for x in cand]);best=cand[np.isclose(ranks,ranks.max())]
        t.append(float(best[np.argmin(np.abs(best-.5))]))
    return np.array(t)

def joint_thresholds(yt,sc):
    # Joint objective protects the rare recipes; 3D search entirely within validation.
    grid=[np.unique(np.r_[.5,np.quantile(sc[:,k],np.linspace(.05,.95,9))]) for k in range(3)]
    true_id=bits_to_id(yt)
    sizes=np.bincount(true_id,minlength=8)
    support=sizes>0
    ismix=yt.sum(1)>=2
    best_key=None;best_t=None
    for tt in product(*grid):
        pred=(sc>=np.array(tt)).astype(np.uint8)
        hits=true_id==bits_to_id(pred)
        per=np.bincount(true_id[hits],minlength=8)/np.maximum(sizes,1)
        macro8=float(per[support].mean())
        exact_mix=float(hits[ismix].mean()) if ismix.any() else 0.0
        # deterministic tie break: macro8, mixture exact, closeness to ordinary .5 threshold
        key=(round(macro8,12),round(exact_mix,12),-float(np.abs(np.array(tt)-.5).sum()))
        if best_key is None or key>best_key: best_key,best_t=key,np.array(tt,dtype=float)
    return best_t

def decode(scores,method,threshold=None,class8=None):
    if method=='argmax8':
        assert class8 is not None
        return BITS[np.argmax(class8,axis=1)].astype(np.uint8)
    return (scores>=threshold).astype(np.uint8)

def oracle_mix_decode(scores,class8=None):
    # DIAGNOSTIC UPPER BOUND: uses knowledge that a sample IS a mixture.
    # Never compare this oracle to deployable models as if the inputs were equivalent.
    codes=np.array([3,5,6,7])
    if class8 is not None:
        return BITS[codes[np.argmax(class8[:,codes],axis=1)]]
    p=np.clip(scores,1e-6,1-1e-6)
    b=BITS[codes]
    logp=(b[None]*np.log(p[:,None])+(1-b)[None]*np.log(1-p[:,None])).sum(2)
    return BITS[codes[np.argmax(logp,axis=1)]]

def recipe_macro_bce(yt,sc):
    p=np.clip(sc,1e-6,1-1e-6)
    errors=-(yt*np.log(p)+(1-yt)*np.log(1-p)).mean(1)
    rid=bits_to_id(yt)
    return float(np.mean([errors[rid==c].mean() for c in np.unique(rid)]))

## 3. Model fitting

Compare ExtraTrees binary relevance and label powerset, ROCKET, CNN, and compact ConvTran with fixed training rules.

In [ ]:
def model_run(name, seed, train_idx, val_idx, evaluation, mode='balanced',fixed_epochs=None):
    """Return scores for named splits and optional eight-class distributions.
    evaluation: {'train':indices, 'validation':indices, 'test':indices}; split separation is caller-controlled.
    """
    seed_everything(seed)
    weights=make_weights(train_idx,mode)
    scores={};class8={};history=[];best_epoch=np.nan
    requested={'train':np.asarray(train_idx),'validation':np.asarray(val_idx),**evaluation}
    if name in ('ExtraTrees3','ExtraTrees8'):
        if name=='ExtraTrees3':
            clfs=[]
            for k in range(3):
                m=ExtraTreesClassifier(n_estimators=300,min_samples_leaf=1,
                      max_features='sqrt',random_state=seed,n_jobs=-1)
                m.fit(Z[train_idx],Y[train_idx,k],sample_weight=weights[train_idx]);clfs.append(m)
            for key,idx in requested.items():
                scores[key]=np.column_stack([m.predict_proba(Z[idx])[:,1] for m in clfs])
        else:
            m=ExtraTreesClassifier(n_estimators=300,min_samples_leaf=1,
                    max_features='sqrt',random_state=seed,n_jobs=-1)
            m.fit(Z[train_idx],recipe[train_idx],sample_weight=weights[train_idx])
            assert np.array_equal(m.classes_,np.arange(8))
            for key,idx in requested.items():
                class8[key]=m.predict_proba(Z[idx]);scores[key]=class8[key]@BITS
    elif name=='ROCKET':
        from sktime.transformations.panel.rocket import Rocket
        # Important: Rocket and scaler fitted on the given train subset only.
        xn=normalize_on_train(train_idx)
        rocket=Rocket(num_kernels=3000,normalise=False,n_jobs=1,random_state=seed)
        trainfeat=np.asarray(rocket.fit_transform(xn[train_idx]),dtype=np.float32)
        scaler=StandardScaler(with_mean=False)
        trainfeat=scaler.fit_transform(trainfeat)
        models=[]
        for k in range(3):
            reg=RidgeClassifier(alpha=1.,solver='lsqr')
            reg.fit(trainfeat,Y[train_idx,k].astype(int),sample_weight=weights[train_idx]);models.append(reg)
        for key,idx in requested.items():
            ft=scaler.transform(np.asarray(rocket.transform(xn[idx]),dtype=np.float32))
            # expit is monotonic only: these are NOT calibrated probabilities.
            scores[key]=np.column_stack([expit(reg.decision_function(ft)) for reg in models])
    elif name in ('CNN','ConvTran'):
        xn=normalize_on_train(train_idx)
        net=(MixtureCNN() if name=='CNN' else ConvTran()).to(DEVICE)
        optimizer=torch.optim.AdamW(net.parameters(),lr=5e-4,weight_decay=1e-4)
        def predict(idx):
            net.eval();outs=[]
            with torch.no_grad():
                for chunk in np.array_split(idx,max(1,int(np.ceil(len(idx)/128)))):
                    z=torch.tensor(xn[chunk],device=DEVICE)
                    outs.append(torch.sigmoid(net(z)).cpu().numpy())
            return np.concatenate(outs)
        best_loss=np.inf; best_state=None;stale=0
        epochs=int(fixed_epochs) if fixed_epochs is not None else MAX_EPOCHS
        for epoch in range(1,epochs+1):
            net.train();sum_weighted=0.;sum_weights=0.
            dataset=TensorDataset(torch.from_numpy(xn[train_idx]),
                                  torch.from_numpy(Y[train_idx]),
                                  torch.from_numpy(weights[train_idx]))
            loader=DataLoader(dataset,batch_size=BATCH,shuffle=True,num_workers=0,
                    generator=torch.Generator().manual_seed(seed+epoch))
            for a,b,w in loader:
                a,b,w=a.to(DEVICE),b.to(DEVICE),w.to(DEVICE)
                optimizer.zero_grad()
                logits=net(a)
                per=nn.functional.binary_cross_entropy_with_logits(logits,b,reduction='none').mean(1)
                loss=(per*w).mean();loss.backward();optimizer.step()
                sum_weighted+=float((per.detach()*w).sum());sum_weights+=float(w.sum())
            train_loss=sum_weighted/sum_weights
            val_pred=predict(val_idx)
            val_loss=recipe_macro_bce(Y[val_idx],val_pred)
            val_fixed=metrics(Y[val_idx],(val_pred>=.5).astype(np.uint8))
            history.append(dict(model=name,seed=seed,mode=mode,epoch=epoch,
                train_weighted_bce=train_loss,val_macro_recipe_bce=val_loss,
                val_macro8_fixed=val_fixed['macro8'],val_exact_mix_fixed=val_fixed['exact_mix']))
            if val_loss<best_loss-1e-5:
                best_loss=val_loss;best_state=copy.deepcopy(net.state_dict());best_epoch=epoch;stale=0
            else:
                stale+=1
                if fixed_epochs is None and stale>=PATIENCE:break
        if best_state is not None:net.load_state_dict(best_state)
        for key,idx in requested.items():scores[key]=predict(idx)
        if fixed_epochs is not None:best_epoch=epochs
        del net
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    else:raise ValueError(name)
    assert all(v.shape==(len(requested[k]),3) and np.isfinite(v).all() for k,v in scores.items())
    return scores,class8,history,best_epoch

## 4. Main evaluation and threshold control

Select thresholds using validation only; compare component-wise and joint decoders without refitting the underlying classifiers.

In [ ]:
# One model fit per predeclared main config; multiple decoders never refit the model.
RUNS=[('ExtraTrees3',42),('ExtraTrees8',42)]+[(m,s) for m in ('ROCKET','CNN','ConvTran') for s in SEEDS]
main_results=[]; prediction_rows=[];threshold_rows=[];history_rows=[];timings=[]
for name,seed in RUNS:
    t0=time.perf_counter()
    scr,cls,hist,best=model_run(name,seed,tr,va,{'test':te},mode='balanced')
    secs=time.perf_counter()-t0
    history_rows+=hist
    methods=('argmax8',) if name=='ExtraTrees8' else ('fixed','perbit','joint')
    per=per_bit_thresholds(Y[va],scr['validation']) if name!='ExtraTrees8' else None
    joint=joint_thresholds(Y[va],scr['validation']) if name!='ExtraTrees8' else None
    for decoder in methods:
        t={'fixed':np.full(3,.5),'perbit':per,'joint':joint}.get(decoder)
        for split,idx in [('train',tr),('validation',va),('test',te)]:
            p=decode(scr[split],decoder,t,cls.get(split))
            met=metrics(Y[idx],p)
            main_results.append(dict(model=name,seed=seed,mode='balanced',decoder=decoder,
                split=split,epochs=best,fit_seconds=secs,**met,**gas_discrimination(Y[idx],scr[split])))
            for j,k in enumerate(idx):
                r=dict(model=name,seed=seed,mode='balanced',decoder=decoder,split=split,
                     sample_id=usable.iloc[k].sample_id,recipe=int(recipe[k]),
                     recipe_key=usable.iloc[k].recipe_key,day=str(pd.Timestamp(usable.iloc[k].time).date()),
                     n_components=int(Y[k].sum()),train_recipe_support=int(audit.iloc[k].train_recipe_support))
                for g_i,g in enumerate(GASES):
                    r[f'true_{g}']=int(Y[k,g_i]);r[f'pred_{g}']=int(p[j,g_i]);
                    r[f'score_{g}']=float(scr[split][j,g_i])
                prediction_rows.append(r)
        if t is not None:
            threshold_rows.extend(dict(model=name,seed=seed,mode='balanced',decoder=decoder,
                              gas=g,threshold=float(t[i])) for i,g in enumerate(GASES))
    # Oracle evaluation is restricted to TRUE mixtures; it is not a deployable model.
    for split,idx in [('train',tr),('validation',va),('test',te)]:
        ismix=Y[idx].sum(1)>=2
        oracle=oracle_mix_decode(scr[split][ismix],cls.get(split)[ismix] if cls else None)
        main_results.append(dict(model=name,seed=seed,mode='balanced',decoder='oracle_known_mixture',
            split=split,epochs=best,fit_seconds=secs,
            **metrics(Y[idx][ismix],oracle)))
    print(f'{name:12s} seed={seed} epochs={best} time={secs:.1f}s  '
          f'val macro8={metrics(Y[va],decode(scr["validation"],methods[-1],joint,cls.get("validation")))["macro8"]:.3f}')

# Non-learning reference strategies, scored with exactly the same metric implementation.
train_mode=int(np.bincount(recipe[tr],minlength=8).argmax())
for name,id_ in [('AlwaysAir',0),('AlwaysAll',7),('TrainMode',train_mode)]:
    for split,idx in [('train',tr),('validation',va),('test',te)]:
        yp=np.tile(BITS[id_],(len(idx),1)).astype(np.uint8)
        main_results.append(dict(model=name,seed=42,mode='constant',decoder='constant',
            split=split,epochs=np.nan,fit_seconds=0.,**metrics(Y[idx],yp)))
        for j,k in enumerate(idx):
            r=dict(model=name,seed=42,mode='constant',decoder='constant',split=split,
                   sample_id=usable.iloc[k].sample_id,recipe=int(recipe[k]),
                   recipe_key=usable.iloc[k].recipe_key,day=str(pd.Timestamp(usable.iloc[k].time).date()),
                   n_components=int(Y[k].sum()),train_recipe_support=int(audit.iloc[k].train_recipe_support))
            for g_i,g in enumerate(GASES):
                r[f'true_{g}']=int(Y[k,g_i]);r[f'pred_{g}']=int(yp[j,g_i]);
                r[f'score_{g}']=np.nan
            prediction_rows.append(r)

MAIN=pd.DataFrame(main_results); PRED=pd.DataFrame(prediction_rows)
HIST=pd.DataFrame(history_rows); THR=pd.DataFrame(threshold_rows)
MAIN.to_csv(OUT/'scores_all_splits.csv',index=False)
PRED.to_csv(OUT/'predictions_all_splits.csv',index=False)
HIST.to_csv(OUT/'training_history.csv',index=False)
THR.to_csv(OUT/'thresholds.csv',index=False)
assert len(MAIN[(MAIN.split=='test')&(MAIN.decoder.isin(['joint','argmax8']))])==len(RUNS)
print('\nMain 146 TEST:')
print(MAIN[(MAIN.split=='test')&(MAIN.decoder.isin(['joint','argmax8']))]
    [['model','seed','decoder','macro8','exact_mix','exact_double','exact_triple']]
    .to_string(index=False,float_format=lambda x:f'{x:.3f}'))

## 5. Localization of errors

Evaluate training, validation, the same-day holdout, recipe support, and oracle mixture-only diagnostics. Oracle results do not represent deployable performance.

In [ ]:
aux=[]
if RUN_SAME_DAY:
    sep27=np.flatnonzero((audit.split=='train') & (audit.day=='2026-09-27'))
    _,hold=train_test_split(sep27,test_size=.2,stratify=recipe[sep27],random_state=2026)
    hold=np.sort(hold)
    diag_train=np.setdiff1d(tr,hold)
    assert len(set(hold)&set(diag_train))==0
    assert set(recipe[hold])==set(range(8))
    print('27 Sep held-out:',len(hold),'mixtures:',int((Y[hold].sum(1)>=2).sum()))
    # The same validation day as main training is used for CNN checkpointing, never the holdout.
    for name,seed in [('ExtraTrees3',42),('ExtraTrees8',42),('CNN',42),('ConvTran',42)]:
        sc,dist,_,epoch=model_run(name,seed,diag_train,va,{'within_day':hold},mode='balanced')
        for split,idx in [('train',diag_train),('within_day',hold),('validation',va)]:
            pred=decode(sc[split],'argmax8' if name=='ExtraTrees8' else 'fixed',
                        np.full(3,.5),dist.get(split))
            aux.append(dict(experiment='within_day',model=name,seed=seed,split=split,
                            epochs=epoch,**metrics(Y[idx],pred)))
    pd.DataFrame(aux).to_csv(OUT/'within_day.csv',index=False)

if RUN_WEIGHT_ABLATION:
    rows=[]
    for name in ('ExtraTrees3','CNN'):
        for mode in ('soft','balanced'):
            ss,cc,_,ep=model_run(name,42,tr,va,{},mode=mode)
            for split,idx in [('train',tr),('validation',va)]:
                yp=decode(ss[split],'fixed',np.full(3,.5),cc.get(split))
                rows.append(dict(model=name,mode=mode,split=split,seed=42,epoch=ep,
                                 **metrics(Y[idx],yp)))
    pd.DataFrame(rows).to_csv(OUT/'weight_ablation.csv',index=False)
    print('Weight ablation:')
    print(pd.DataFrame(rows)[['model','mode','split','macro8','exact_mix']].to_string(index=False))

# Negative control: destroy the train signal-label association; a strong result would be suspicious.
# Keep labels/weights attached to each other when shuffling (otherwise weights leak the original class).
rng=np.random.default_rng(2033)
shuffled_idx=rng.permutation(tr)
shuffled_weights=make_weights(tr,'balanced')[shuffled_idx]
null_models=[]
for gidx in range(3):
    null=ExtraTreesClassifier(n_estimators=300,min_samples_leaf=1,max_features='sqrt',
                              random_state=42,n_jobs=-1)
    null.fit(Z[tr],Y[shuffled_idx,gidx].astype(int),sample_weight=shuffled_weights)
    null_models.append(null)
null_val=np.column_stack([q.predict_proba(Z[va])[:,1] for q in null_models])
null_prediction=(null_val>=.5).astype(np.uint8)
pd.DataFrame([dict(control='permuted_train_labels_ET3',split='validation',
                   **metrics(Y[va],null_prediction))]).to_csv(OUT/'negative_control.csv',index=False)
print('Shuffled-label negative control, val macro8:',round(metrics(Y[va],null_prediction)['macro8'],3))

# Oracle mixture-only 1-NN: source train mixtures only; one-nearest-neighbor train LOO.
from sklearn.metrics import pairwise_distances
trm=tr[Y[tr].sum(1)>=2]
scaler=StandardScaler().fit(Z[trm])
base=scaler.transform(Z[trm]); NN=NearestNeighbors(n_neighbors=2).fit(base)
knn_rows=[]
for split,idx in [('train_LOO',trm),('validation',va[Y[va].sum(1)>=2]),
                  ('test',te[Y[te].sum(1)>=2])]:
    query=scaler.transform(Z[idx]); neigh=NN.kneighbors(query,n_neighbors=2 if split=='train_LOO' else 1,
                                                    return_distance=False)
    neighbor=neigh[:,1] if split=='train_LOO' else neigh[:,0]
    yp=Y[trm][neighbor].astype(np.uint8)
    knn_rows.append(dict(experiment='oracle_mix_1NN',model='1NN-true-mixture',split=split,
                         **metrics(Y[idx],yp)))
pd.DataFrame(knn_rows).to_csv(OUT/'oracle_mixture_1nn.csv',index=False)
print('Mixture-only nearest-neighbor:',pd.DataFrame(knn_rows)[['split','exact_mix']].to_string(index=False))

## 6. Memorization check

Test whether the neural architectures can fit a small fixed balanced subset. This is a debugging control, not a future-test result.

In [ ]:
if RUN_MEMORIZATION:
    probe_idx=np.concatenate([np.random.default_rng(2026+c).choice(tr[recipe[tr]==c],4,replace=False)
                              for c in range(8)])
    assert len(probe_idx)==32 and np.all(np.bincount(recipe[probe_idx])==4)
    probe_rows=[]; curve_rows=[]
    for name in ('CNN','ConvTran'):
        seed_everything(42)
        xp=normalize_on_train(probe_idx)
        m=(MixtureCNN() if name=='CNN' else ConvTran()).to(DEVICE)
        optim=torch.optim.AdamW(m.parameters(),lr=1e-3,weight_decay=0)
        a=torch.tensor(xp[probe_idx],device=DEVICE)
        y=torch.tensor(Y[probe_idx],device=DEVICE)
        # Balanced subset -> unweighted loss. Keep dropout/BN exactly as architecture defines.
        reached=False;epoch=0
        for epoch in range(1,201):
            m.train();optim.zero_grad()
            logits=m(a)
            loss=nn.functional.binary_cross_entropy_with_logits(logits,y)
            loss.backward();optim.step()
            m.eval()
            with torch.no_grad(): pp=torch.sigmoid(m(a)).cpu().numpy()
            mm=metrics(Y[probe_idx],(pp>=.5).astype(np.uint8))
            curve_rows.append(dict(model=name,epoch=epoch,train_exact=mm['exact_all'],
                                   train_macro8=mm['macro8'],train_loss=float(loss.item())))
            if mm['exact_all']>=.99:
                reached=True;break
        probe_rows.append(dict(model=name,n=32,epoch=epoch,train_exact=mm['exact_all'],
                               train_macro8=mm['macro8'],perfect_fit=reached))
        del m
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    pd.DataFrame(probe_rows).to_csv(OUT/'memorization_probe.csv',index=False)
    pd.DataFrame(curve_rows).to_csv(OUT/'memorization_curve.csv',index=False)
    print('Neural memorization:',pd.DataFrame(probe_rows).to_string(index=False))

## 7. Exported diagnostics

Export composition confusion, component recall/specificity, predicted cardinality, and uncertainty across training seeds.

In [ ]:
# Error tables, gas errors, predicted cardinality, composition confusion.
from sklearn.metrics import confusion_matrix
from collections import Counter
pergas=[];pred_card=[];comp_conf=[];support_rows=[]
for (name,seed,decoder,split),g in PRED.groupby(['model','seed','decoder','split'],sort=False):
    yt=g[[f'true_{s}' for s in GASES]].to_numpy(dtype=np.uint8)
    yp=g[[f'pred_{s}' for s in GASES]].to_numpy(dtype=np.uint8)
    ntrue=yt.sum(1); npred=yp.sum(1)
    for k,gas in enumerate(GASES):
        a,b=yt[:,k],yp[:,k]
        tp=int(((a==1)&(b==1)).sum());tn=int(((a==0)&(b==0)).sum())
        fp=int(((a==0)&(b==1)).sum());fn=int(((a==1)&(b==0)).sum())
        pergas.append(dict(model=name,seed=seed,decoder=decoder,split=split,
            gas=gas,TP=tp,TN=tn,FP=fp,FN=fn,
            recall=tp/max(tp+fn,1),specificity=tn/max(tn+fp,1)))
    for a in range(4):
        for b in range(4):
            pred_card.append(dict(model=name,seed=seed,decoder=decoder,split=split,
                true_n=a,pred_n=b,count=int(((ntrue==a)&(npred==b)).sum())))
    cm=confusion_matrix(bits_to_id(yt),bits_to_id(yp),labels=range(8))
    for i,j in product(range(8),repeat=2):
        comp_conf.append(dict(model=name,seed=seed,decoder=decoder,split=split,
                              true_id=i,pred_id=j,count=int(cm[i,j])))
    if split in ('validation','test'):
        correct=(yt==yp).all(1)
        for label,condition in [('unseen',g.train_recipe_support.eq(0)),
                                ('1-2',g.train_recipe_support.between(1,2)),
                                ('3+',g.train_recipe_support.ge(3))]:
            z=condition.to_numpy() & (ntrue>=2)
            support_rows.append(dict(model=name,seed=seed,decoder=decoder,split=split,
                                     support=label,n=int(z.sum()),
                                     mixture_exact=float(correct[z].mean()) if z.any() else np.nan))

for name,table in [('per_gas.csv',pergas),('cardinality_confusion.csv',pred_card),
                   ('eight_recipe_confusion.csv',comp_conf),('recipe_support_errors.csv',support_rows)]:
    pd.DataFrame(table).to_csv(OUT/name,index=False)
# Pairwise correctness on the SAME exposures (descriptive only: all mixtures on one day).
pair_rows=[]
for split in ('validation','test'):
    def frame(model):
        a=PRED[(PRED.model==model)&(PRED.seed==42)&(PRED.decoder=='joint')&(PRED.split==split)].copy()
        a['correct']=a.apply(lambda r: all(int(r[f'true_{gas}'])==int(r[f'pred_{gas}']) for gas in GASES),axis=1)
        return a[['sample_id','n_components','correct']]
    left=frame('ExtraTrees3');right=frame('CNN')
    pair=left.merge(right,on=['sample_id','n_components'],validate='one_to_one',suffixes=('_ET','_CNN'))
    pair=pair[pair.n_components>=2]
    pair_rows.append(dict(split=split,n=len(pair),both_correct=int((pair.correct_ET&pair.correct_CNN).sum()),
        ET_only=int((pair.correct_ET&~pair.correct_CNN).sum()),
        CNN_only=int((~pair.correct_ET&pair.correct_CNN).sum()),
        both_wrong=int((~pair.correct_ET&~pair.correct_CNN).sum())))
pd.DataFrame(pair_rows).to_csv(OUT/'paired_mixture_errors.csv',index=False)
# Fixed publication table; no best-of-test selection.
primary=MAIN[MAIN.decoder.isin(['joint','argmax8','constant'])].copy()
summary=(primary.groupby(['model','split'],as_index=False)
         .agg(n_runs=('seed','size'),macro8=('macro8','mean'),
              macro8_sd=('macro8','std'),mix_exact=('exact_mix','mean'),
              mix_sd=('exact_mix','std'),double=('exact_double','mean'),
              triple=('exact_triple','mean'),pergas_bacc=('macro_bacc','mean')))
summary.to_csv(OUT/'E5_diagnostic_summary.csv',index=False)
rows=['\\begin{table*}[t]', '\\centering\\small',
      '\\caption{Exploratory E5 diagnostics: fixed train/validation split on campaign 130 and previously examined campaign 146. Macro-8 is macro recall over all eight compositions; mixture exact is evaluated on true mixtures only. Means over seeds are not independent exposure-day replicates.}',
      '\\label{tab:e5diag}', '\\begin{tabular}{lrrrrrr}', '\\toprule',
      'Model & Val macro-8 & Val mix exact & Test macro-8 & Test mix exact & Test double & Test triple \\\\', '\\midrule']
for name in list(METHODS)+['AlwaysAll','AlwaysAir','TrainMode']:
    v=summary[(summary.model==name)&(summary.split=='validation')].iloc[0]
    t=summary[(summary.model==name)&(summary.split=='test')].iloc[0]
    def pp(z):return '--' if not np.isfinite(z) else f'{100*z:.1f}'
    rows.append(f'{name} & {pp(v.macro8)} & {pp(v.mix_exact)} & {pp(t.macro8)} & '
                f'{pp(t.mix_exact)} & {pp(t.double)} & {pp(t.triple)} '+r'\\')
rows += ['\\bottomrule','\\end{tabular}','\\end{table*}']
(OUT/'E5_diagnostic_table.tex').write_text('\n'.join(rows)+'\n',encoding='utf-8')
print(summary.to_string(index=False,float_format=lambda x:f'{x:.3f}'))

In [ ]:
# Compact, predeclared plots: ExtraTrees3, balanced weights, joint decoder, seed 42.
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
key=PRED[(PRED.model=='ExtraTrees3')&(PRED.seed==42)&(PRED.decoder=='joint')]
with PdfPages(OUT/'E5_diagnostic_plots.pdf') as pdf:
    fig,axes=plt.subplots(1,2,figsize=(9.2,3.7))
    for ax,split in zip(axes,['validation','test']):
        d=key[key.split==split]
        y=d[[f'true_{x}' for x in GASES]].to_numpy(dtype=int)
        p=d[[f'pred_{x}' for x in GASES]].to_numpy(dtype=int)
        cm=confusion_matrix(bits_to_id(y),bits_to_id(p),labels=range(8))
        ax.imshow(cm,cmap='Blues')
        ax.set_title(f'ExtraTrees3 · {split} (n={len(d)})')
        ax.set_xlabel('Predicted composition (0..7)')
        ax.set_ylabel('True composition (0..7)')
        ax.set_xticks(range(8));ax.set_yticks(range(8))
    fig.tight_layout();pdf.savefig(fig);plt.close(fig)
    h=HIST[HIST.model.isin(['CNN','ConvTran'])]
    if len(h):
        fig,axes=plt.subplots(1,2,figsize=(9.2,3.5))
        for ax,name in zip(axes,['CNN','ConvTran']):
            for seed,g in h[h.model==name].groupby('seed'):
                ax.plot(g.epoch,g.val_macro_recipe_bce,label=f'seed {seed}')
            ax.set_title(f'{name}: validation recipe-balanced BCE')
            ax.set_xlabel('Epoch');ax.set_ylabel('Loss');ax.legend(fontsize=8)
        fig.tight_layout();pdf.savefig(fig);plt.close(fig)

# Physically interpretable comparison at ONE exact nominal recipe (concentrations held fixed).
# Choose only by sample support, never by prediction quality.
mixmeta=audit[audit.n_components>=2].copy()
support=mixmeta.groupby(['recipe_key','split']).size().unstack(fill_value=0)
for split in ('train','validation','test'):
    if split not in support.columns:support[split]=0
candidates=support[(support[['train','validation','test']]>0).all(axis=1)].copy()
if len(candidates):
    candidates['min_n']=candidates[['train','validation','test']].min(axis=1)
    candidates['sum_n']=candidates[['train','validation','test']].sum(axis=1)
    chosen=candidates.sort_values(['min_n','sum_n'],ascending=False).index[0]
    (OUT/'illustrated_recipe.txt').write_text(str(chosen)+'\n',encoding='utf-8')
    fig,axes=plt.subplots(1,2,figsize=(9.0,3.2))
    T=np.linspace(175,390,200)
    for ax,component,label in [(axes[0],0,'R1: heating'),(axes[1],6,'R1: hysteresis')]:
        for split in ('train','validation','test'):
            ii=np.flatnonzero((audit.recipe_key==chosen)&(audit.split==split))
            yy=X[ii,component]
            median=np.median(yy,axis=0)
            low=np.percentile(yy,25,axis=0);high=np.percentile(yy,75,axis=0)
            ax.plot(T,median,label=f'{split} (n={len(ii)})')
            ax.fill_between(T,low,high,alpha=.13)
        ax.set_title(label);ax.set_xlabel('Temperature (C)')
        ax.set_ylabel('Referenced log-conductance units')
    axes[0].legend(fontsize=8)
    fig.suptitle('Identical nominal recipe across three cohorts; medians and IQR',fontsize=10)
    fig.tight_layout();fig.savefig(OUT/'E5_fixed_recipe_signals.pdf',bbox_inches='tight');plt.close(fig)

# Predeclared diagnostic rules. They produce hypotheses, not causal proof.
report=['# E5 diagnostic interpretation (exploratory)','',
    'All test analyses refer to the already inspected October 8 mixtures (one test day).',
    'Macro-8 and mixture exact match are distinct; check both, including the always-all baseline.', '']
for name,seed in [('ExtraTrees3',42),('CNN',42),('ConvTran',42)]:
    sub=MAIN[(MAIN.model==name)&(MAIN.seed==seed)&(MAIN.decoder=='joint')]
    train=sub[sub.split=='train'].iloc[0];val=sub[sub.split=='validation'].iloc[0]
    test=sub[sub.split=='test'].iloc[0]
    oracle=MAIN[(MAIN.model==name)&(MAIN.seed==seed)&(MAIN.decoder=='oracle_known_mixture')&
                (MAIN.split=='test')].iloc[0]
    report.append(f'## {name} (seed {seed})')
    report.append(f'- Train mixture exact: {train.exact_mix:.3f}; Sept 28 val: {val.exact_mix:.3f}; '
                  f'Oct 8 test: {test.exact_mix:.3f}.')
    report.append(f'- Test mixture-conditioned oracle exact: {oracle.exact_mix:.3f} '
                  '(not deployable).')
    if train.exact_mix < .6:
        report.append('- **Hypothesis:** poor fit even on observed training mixtures; inspect class weighting, optimization, capacity and overlapping signals.')
    elif val.exact_mix < train.exact_mix-.25:
        report.append('- **Hypothesis:** generalization gap already within campaign 130; inspect recipe support and date composition.')
    else:report.append('- No large train-to-validation mixture exact gap by the stated heuristic.')
    if oracle.exact_mix > test.exact_mix+.2:
        report.append('- **Hypothesis:** mixture-detection/gating or threshold decisions contribute markedly; oracle assumes true mixture identity.')
    report.append('')
report+=['## Interpretation constraints',
  '- If the same-day holdout is strong but next-day weak, temporal/recipe distribution differences are plausible, not proven sensor drift.',
  '- If 32-example memorization fails, first inspect optimization and implementation; success says nothing about cross-day generalization.',
  '- If 1-NN mixture-only performs well but end-to-end models fail, evaluate class imbalance and mixture gating before claiming physical indistinguishability.',
  '- If AUROC is high but exact match is low, thresholds/ranking should be investigated; do not optimize on test.',
  '- Small per-recipe samples and one mixture test day prevent a population-level confidence interval.',
  '- E5 is an exploratory reanalysis; avoid replacing the original E5 result as if it were a new untouched benchmark.']
(OUT/'E5_diagnosis.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
(OUT/'run_config.json').write_text(json.dumps(dict(archives=ARCHIVES,cache_signature=cache_signature,
    channels=CHANNELS,gases=GASES,seeds=SEEDS,methods=METHODS,baselines=['AlwaysAir','AlwaysAll','TrainMode'],
    weighting='balanced 8 compositions, full inverse training counts',
    validation_selection='macro-recipe BCE; no test in checkpoint or thresholds',
    threshold_main='joint macro-8 recall, tie mixture exact on validation',
    primary_metrics=['macro8','exact_mix','exact_double','exact_triple'],
    preregistration='exploratory AFTER inspection of original E5 test',
    neural_epochs=MAX_EPOCHS,patience=PATIENCE,
    same_day=RUN_SAME_DAY,memorization=RUN_MEMORIZATION,
    weight_ablation=RUN_WEIGHT_ABLATION,
    sklearn=__import__('sklearn').__version__,torch=torch.__version__,device=str(DEVICE)),
    ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved complete diagnostics to',OUT)